# 🏭 Industrial Predictive Maintenance & Fault Diagnostic Engine
## Industrial Predictive Maintenance & Fault Diagnostic Engine

---

# Phase 1: Problem & Data Framing

This is the first of the six **CRISP-DM** phases. Its goal is to lay a **solid foundation** before writing any modeling code, because every decision we make later is built on it.

## 🎯 What we build

A system that predicts machine failure **before it happens** from live sensor readings, then **diagnoses the failure type**. These are **two tasks** built on the same data:

| Task | Type | Target |
|---------|-------|-------|
| Will the machine fail? | Binary classification | `Machine failure` = 0/1 |
| What failure type? | Multi-class classification | `Failure Type` = TWF/HDF/PWF/OSF/RNF |

## 📦 Dataset

We use the **AI4I 2020 Predictive Maintenance Dataset** from Kaggle, an academic/industrial benchmark simulating a milling machine with 10,000 records and the following sensors:

- `Air temperature [K]` — ambient air temperature
- `Process temperature [K]` — manufacturing process temperature
- `Rotational speed [rpm]` — rotational speed
- `Torque [Nm]` — torque
- `Tool wear [min]` — cutting tool wear

> 💡 **Kaggle path:** `/kaggle/input/ai4i-predictive-maintenance-dataset/ai4i2020.csv`

## 🛠️ Quality rules we commit to throughout the project

1. **No magic numbers:** every constant is defined once at the top of the notebook and used by name.
2. **Documented functions:** every piece of logic is wrapped in a function with a clear name + docstring + type hints.
3. **Data quality checks:** we verify consistency before trusting the data.
4. **Reproducibility:** a fixed random seed `RANDOM_STATE`.


## 1.1 — Setup: imports and central constants

**What did we use and why?**

| Element | Reason |
|--------|-------|
| `pathlib.Path` | Handle paths as objects instead of strings — more precise and safer across platforms |
| `RANDOM_STATE = 42` | Fix the random seed so all results are **fully reproducible** |
| Column constants | Single Source of Truth — if a column name changes, we edit it in one place only |

> ⚠️ Note: the column names here contain **spaces and square brackets**, so they cannot be accessed via `df.col_name` but via `df["column name"]`.


In [ ]:
# ============================================================
# Phase 1 — Setup and constants
# ============================================================
import numpy as np
import pandas as pd
from pathlib import Path

# ─── General constants ────────────────────────────────────
RANDOM_STATE = 42          # random seed: ensures reproducibility
np.random.seed(RANDOM_STATE)

# ─── Data paths (Kaggle) ─────────────────────────────────
DATA_DIR  = Path("/kaggle/input/ai4i-predictive-maintenance-dataset")
DATA_FILE = DATA_DIR / "ai4i2020.csv"

# ─── Column groups ────────────────────────────────────────
# Identifier columns (carry no physical information for the model)
ID_COLUMNS = ["UDI", "Product ID"]

# Sensor columns = physical features
SENSOR_COLUMNS = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

# Binary target
BINARY_TARGET = "Machine failure"

# Failure-type columns (encoded as 0/1 flags inside the file)
FAILURE_TYPE_COLUMNS = ["TWF", "HDF", "PWF", "OSF", "RNF"]

# Column we will derive for the multi-class task
MULTI_CLASS_TARGET = "Failure Type"

print("Constants ready ✅")


## 1.2 — Load data function

**Why wrap loading in a function instead of writing it directly?**

1. **Reuse:** we will call it in later phases without repetition.
2. **Error handling:** we check the file exists and give a clear message instead of a cryptic crash.
3. **Documentation:** the docstring explains inputs and outputs to anyone reading the code later (including you, a month from now!).

> Note: `Type` here is the **product quality** (Low/Medium/High), an important categorical variable we will study in Phase 3.


In [ ]:
def load_data(path: Path) -> pd.DataFrame:
    """
    Load the predictive maintenance dataset from a CSV file.

    Parameters
    ----------
    path : Path
        Full path to ai4i2020.csv.

    Returns
    -------
    pd.DataFrame
        Raw data as-is (no preprocessing).

    Raises
    ------
    FileNotFoundError
        If the file does not exist at the given path.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"File not found: {path}\n"
            "Make sure the 'AI4I 2020 Predictive Maintenance' dataset is added to your Kaggle notebook."
        )
    return pd.read_csv(path)

# ─── Load the data ────────────────────────────────────────
df = load_data(DATA_FILE)
print(f"Loaded successfully: {df.shape[0]:,} rows × {df.shape[1]} columns")


## 1.3 — Build both targets (binary and multi-class)

Surprise: the dataset does **not** contain a "Failure Type" column directly! Instead it contains:
- A binary column `Machine failure` (did a failure occur?).
- Five flag columns `TWF, HDF, PWF, OSF, RNF` (failure type, encoded as 0/1 per type).

So we **derive** the `Failure Type` column ourselves. But during inspection we discovered a **documented internal inconsistency** in this data:

> ⚠️ The failure-type flags are generated from **physical rules** (e.g. temp diff < 8.6K and speed < 1380 rpm → HDF), while the `Machine failure` column was set independently. Result: some rows carry a failure type (e.g. `HDF=1`) but `Machine failure=0`.

**The policy we adopt (professionally correct):**

1. **Binary target** stays `Machine failure` — the official truth for overall failure.
2. **Failure type** is derived only from rows that actually failed (`Machine failure == 1`).
3. **Co-occurring failures** are joined with `+` (e.g. `TWF+HDF`).
4. We document the mismatch size instead of crashing with `assert`.

**Meaning of the failure types:**
| Code | Meaning |
|-------|--------|
| TWF | Tool Wear Failure |
| HDF | Heat Dissipation Failure |
| PWF | Power Failure |
| OSF | Overstrain Failure |
| RNF | Random Failure |


In [ ]:
def build_targets(df: pd.DataFrame) -> pd.DataFrame:
    """
    Derive the multi-class target while reconciling the known
    inconsistency in the AI4I dataset.

    Important note: in the original AI4I data, the 'Machine failure' column
    does not exactly match the failure-type flags; some rows carry a failure
    type without the overall failure flag. Here we treat 'Machine failure'
    as the ground truth for overall failure, and derive the failure type
    only from rows that actually failed.
    """
    df = df.copy()
    flag_sum = df[FAILURE_TYPE_COLUMNS].sum(axis=1)

    # ── Diagnose the mismatch (understand before fixing) ─────────
    type_without_failure = (flag_sum >= 1) & (df[BINARY_TARGET] == 0)
    failure_without_type = (flag_sum == 0) & (df[BINARY_TARGET] == 1)
    print(f"🔎 Rows with a failure type but no Machine failure: {type_without_failure.sum()}")
    print(f"🔎 Rows with Machine failure but no failure type : {failure_without_type.sum()}")

    # ── Policy ─────────────────────────────────
    # 1) Binary target stays: 'Machine failure' (ground truth).
    # 2) Failure type is derived only from failed rows (Machine failure == 1).
    # 3) Co-occurring failures are joined with '+' (e.g. TWF+HDF).
    df[MULTI_CLASS_TARGET] = "No Failure"
    failed_mask = df[BINARY_TARGET] == 1
    df.loc[failed_mask, MULTI_CLASS_TARGET] = (
        df.loc[failed_mask, FAILURE_TYPE_COLUMNS]
        .apply(lambda row: "+".join(row.index[row == 1]) or "Unknown", axis=1)
    )
    return df

# ─── Apply the derivation ────────────────────────────────────────
df = build_targets(df)

# Quick look at the result and type distribution (including co-occurring)
print("\nFailure type distribution:")
print(df[MULTI_CLASS_TARGET].value_counts().to_string())


## 1.4 — Data overview

Finally we print a **data overview**: dimensions, types, missing values, and target distributions. This gives us a first picture that reveals:

- The size of **class imbalance** — to be handled later in Phase 3.
- Whether any values are missing from the start.


In [ ]:
def data_overview(df: pd.DataFrame) -> None:
    """
    Print a comprehensive summary of the data (shape, dtypes, missing values, target distributions).
    """
    print("=" * 55)
    print("📊 Data Overview")
    print("=" * 55)
    print(f"Rows    : {df.shape[0]:,}")
    print(f"Columns : {df.shape[1]}")

    print("\n── Column dtypes ──")
    print(df.dtypes.value_counts().to_string())

    print("\n── Missing values ──")
    missing = df.isnull().sum()
    print("No missing values ✅" if missing.sum() == 0 else missing[missing > 0])

    print("\n── Binary target distribution (Machine failure) ──")
    print(df[BINARY_TARGET].value_counts().to_string())
    print(f"Failure rate: {df[BINARY_TARGET].mean() * 100:.2f}%")

    print("\n── Failure type distribution ──")
    print(df[MULTI_CLASS_TARGET].value_counts().to_string())

# ─── Call the overview ───────────────────────────────────────
data_overview(df)


## ✅ Phase 1 summary

We accomplished the following:

1. **Defined the two tasks** (binary and multi-class) and the goal of each.
2. **Established a central constants structure** to be used in all later phases.
3. **Wrote documented functions** for loading and building targets.
4. **Discovered a documented internal inconsistency** between the failure flags and the `Machine failure` column, and adopted a clear, justified reconciliation policy.
5. **Confirmed severe class imbalance** (failure rate ~3.4%) — this will drive our decisions in Phase 3 (Stratified K-Fold + `scale_pos_weight`).

### 🔜 Next: Phase 2 — Exploratory Data Analysis (EDA)
We will analyze the distribution of each sensor, plot the correlation matrix, and examine how sensor readings differ between healthy and failed machines.
